<a href="https://colab.research.google.com/github/donaldng05/carla/blob/main/notebooks/phase3_behavioral_cloning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3 — Behavioral Cloning

This notebook is the Colab orchestration layer for the camera-only ResNet-18 + Transformer behavioral-cloning experiment. Production code remains under `src/`; this notebook records configuration, bounded dataset inspection, training, evaluation, and artifacts.

## 1. Colab environment and reproducible source

Run this notebook in a fresh Colab runtime with a T4 or better. The branch is cloned at the beginning so the run is tied to an exact repository revision.

In [3]:
import os
import sys
import json
import random
import shutil
import subprocess
import platform
from pathlib import Path

REPO_URL = os.environ.get(
    "CARLA_REPO_URL",
    "https://github.com/donaldng05/carla.git",
)
REPO_ROOT = Path("/content/carla")

def run_git(args):
    return subprocess.run(
        ["git", *args],
        check=True,
        text=True,
        capture_output=True,
    )

# Remove an incomplete or failed directory to ensure a clean clone
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

# Clone without specifying a branch to auto-detect the default one
print(f"Cloning {REPO_URL}...")
run_git([
    "clone",
    "--single-branch",
    REPO_URL,
    str(REPO_ROOT),
])

# Detect which branch was actually cloned
BRANCH = run_git(["-C", str(REPO_ROOT), "rev-parse", "--abbrev-ref", "HEAD"]).stdout.strip()

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO_ROOT / "requirements-dev.txt"),
    ],
    check=True,
)

os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True,
).strip()

print("repository:", REPO_ROOT)
print("detected branch:", BRANCH)
print("commit:", COMMIT)

Cloning https://github.com/donaldng05/carla.git...


CalledProcessError: Command '['git', 'clone', '--single-branch', 'https://github.com/donaldng05/carla.git', '/content/carla']' returned non-zero exit status 128.

In [ ]:
import numpy as np
import torch, torchvision

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('python:', platform.python_version())
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('device:', DEVICE)
if DEVICE.type == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))
    print('gpu_memory_gb:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print('WARNING: CUDA is unavailable; do not start the full training run.')

## 2. Configuration and run metadata

In [ ]:
from src.planning.train_bc import load_simple_yaml

CONFIG = load_simple_yaml(REPO_ROOT / 'configs' / 'bc_v1.yaml')
CONFIG['seed'] = SEED
CONFIG['device'] = str(DEVICE)
CONFIG['commit'] = COMMIT
CONFIG['train_limit'] = 50000
CONFIG['validation_limit'] = 8400
CONFIG['test_limit'] = 7200
CONFIG['smoke_limit'] = 12
CONFIG['dataset_split'] = {'train': 'train', 'validation': 'validation', 'test': 'test'}
RUN_DIR = Path(CONFIG.get('output_dir', 'outputs/phase3')) / f'run_{COMMIT[:8]}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'resolved_config.json').write_text(json.dumps(CONFIG, indent=2, default=str))
print(json.dumps(CONFIG, indent=2, default=str))

## 3. Bounded dataset inspection

The source dataset is streamed. The notebook materializes only bounded split windows; it never downloads the full dataset. Increase the limits only after the smoke test succeeds and Colab memory is confirmed.

In [ ]:
from itertools import islice
from datasets import load_dataset

DATASET_NAME = CONFIG['dataset_name']
def stream_samples(split, limit):
    stream = load_dataset(DATASET_NAME, split=split, streaming=True)
    return list(islice(stream, limit))

inspection = stream_samples('validation', 40)
frames = [row.get('frame') for row in inspection if isinstance(row.get('frame'), (int, float))]
timestamps = [row.get('timestamp') for row in inspection if isinstance(row.get('timestamp'), (int, float))]
steers = [float(row.get('steer', 0.0)) for row in inspection]
throttles = [float(row.get('throttle', 0.0)) for row in inspection]
spacing = [b-a for a,b in zip(frames, frames[1:])]
time_spacing = [b-a for a,b in zip(timestamps, timestamps[1:])]
inspection_summary = {
    'sample_count': len(inspection),
    'run_ids': sorted({row.get('run_id') for row in inspection}),
    'frame_delta_values': sorted(set(spacing)),
    'timestamp_delta_min': min(time_spacing) if time_spacing else None,
    'timestamp_delta_max': max(time_spacing) if time_spacing else None,
    'steer_range': [min(steers), max(steers)],
    'throttle_range': [min(throttles), max(throttles)],
    'image_shape': list(np.asarray(inspection[0]['image_front']).shape),
}
(RUN_DIR / 'inspection_summary.json').write_text(json.dumps(inspection_summary, indent=2))
print(json.dumps(inspection_summary, indent=2))

## 4. Golden-data smoke test

In [ ]:
from torch.utils.data import DataLoader
from src.data.bc_sequences import BehavioralCloningSequenceDataset, collate_bc_sequences
from src.data.carla_dataset import build_bc_preprocessing_config
from src.planning.bc_model import BehavioralCloningModel
from src.planning.bc_training import run_bc_epoch, save_checkpoint, load_checkpoint

golden_raw = inspection[:CONFIG['smoke_limit']]
golden_ds = BehavioralCloningSequenceDataset(golden_raw, sequence_length=CONFIG['sequence_length'], preprocessing=build_bc_preprocessing_config())
assert len(golden_ds) >= 1, 'inspection window did not contain enough contiguous frames'
golden_loader = DataLoader(golden_ds, batch_size=2, shuffle=False, collate_fn=collate_bc_sequences)
golden_model = BehavioralCloningModel(sequence_length=CONFIG['sequence_length'], pretrained=True).to(DEVICE)
golden_optimizer = torch.optim.AdamW(golden_model.parameters(), lr=1e-4)
golden_before = run_bc_epoch(golden_model, golden_loader, golden_optimizer, device=DEVICE, augmentation_shift_pixels=0)
golden_after = run_bc_epoch(golden_model, golden_loader, golden_optimizer, device=DEVICE, augmentation_shift_pixels=0)
assert next(iter(golden_loader))['bc_target'].shape[-1] == 2
assert golden_after['loss'] <= golden_before['loss'] or len(golden_ds) < 2
golden_ckpt = RUN_DIR / 'golden_checkpoint.pt'
save_checkpoint(golden_ckpt, epoch=1, model=golden_model, optimizer=golden_optimizer, scheduler=None, best_val_loss=golden_after['loss'], config=CONFIG)
reloaded = BehavioralCloningModel(sequence_length=CONFIG['sequence_length'], pretrained=False).to(DEVICE)
reloaded_optimizer = torch.optim.AdamW(reloaded.parameters(), lr=1e-4)
load_checkpoint(golden_ckpt, model=reloaded, optimizer=reloaded_optimizer, map_location=DEVICE)
print('golden smoke passed:', golden_before, golden_after, golden_ckpt)

## 5. Bounded train/validation/test sequence datasets

This is the memory-safe first training path. It keeps only the configured bounded sample count in memory while preserving source order for sequence construction. Run-level split isolation comes from the upstream dataset splits and is audited below.

In [ ]:
preprocessing = build_bc_preprocessing_config()
train_raw = stream_samples('train', CONFIG['train_limit'])
validation_raw = stream_samples('validation', CONFIG['validation_limit'])
test_raw = stream_samples('test', CONFIG['test_limit'])
train_ds = BehavioralCloningSequenceDataset(train_raw, sequence_length=CONFIG['sequence_length'], preprocessing=preprocessing)
validation_ds = BehavioralCloningSequenceDataset(validation_raw, sequence_length=CONFIG['sequence_length'], preprocessing=preprocessing)
test_ds = BehavioralCloningSequenceDataset(test_raw, sequence_length=CONFIG['sequence_length'], preprocessing=preprocessing)
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_bc_sequences, num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
validation_loader = DataLoader(validation_ds, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_bc_sequences, num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_bc_sequences, num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
split_summary = {name: {'raw_samples': len(raw), 'windows': len(ds), 'run_ids': len({row.get('run_id') for row in raw})} for name, raw, ds in [('train', train_raw, train_ds), ('validation', validation_raw, validation_ds), ('test', test_raw, test_ds)]}
(RUN_DIR / 'split_summary.json').write_text(json.dumps(split_summary, indent=2))
print(json.dumps(split_summary, indent=2))

## 6. Model, training, and Drive checkpoints

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_RUN_DIR = Path('/content/drive/MyDrive/carla_bc') / RUN_DIR.name
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
CONFIG['checkpoint_dir'] = str(DRIVE_RUN_DIR)
(DRIVE_RUN_DIR / 'resolved_config.json').write_text(json.dumps(CONFIG, indent=2, default=str))

from src.planning.bc_model import build_bc_model
from src.planning.train_bc import train_model
try:
    import wandb
    WANDB_RUN = wandb.init(project='carla-bc', config=CONFIG, reinit='create_new')
except Exception as exc:
    wandb, WANDB_RUN = None, None
    print('W&B disabled:', exc)

model = build_bc_model(CONFIG).to(DEVICE)
trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in model.parameters())
print(f'trainable parameters: {trainable:,} / {total:,}')

CONFIG['output_dir'] = str(DRIVE_RUN_DIR)
history = train_model(model, train_loader, validation_loader, CONFIG, device=DEVICE, checkpoint_dir=DRIVE_RUN_DIR)
(RUN_DIR / 'training_history.json').write_text(json.dumps(history, indent=2))
(DRIVE_RUN_DIR / 'training_history.json').write_text(json.dumps(history, indent=2))
if wandb is not None:
    for record in history:
        wandb.log(record)
    wandb.finish()
print('training complete; epochs:', len(history))

## 7. Held-out evaluation and qualitative predictions

In [ ]:
from src.planning.bc_evaluation import action_metrics
from src.planning.bc_training import load_checkpoint, run_bc_epoch

best_model = build_bc_model(CONFIG).to(DEVICE)
best_state = load_checkpoint(DRIVE_RUN_DIR / 'best_checkpoint.pt', model=best_model, map_location=DEVICE)
with torch.no_grad():
    test_metrics = run_bc_epoch(best_model, test_loader, None, device=DEVICE, steering_weight=CONFIG['steering_loss_weight'], throttle_weight=CONFIG['throttle_loss_weight'])
test_metrics.update({'checkpoint_epoch': best_state['epoch'], 'effective_temporal_horizon_seconds': test_ds[0]['metadata']['temporal_horizon_seconds'], 'skipped_windows': 0})
(RUN_DIR / 'test_metrics.json').write_text(json.dumps(test_metrics, indent=2))
print(json.dumps(test_metrics, indent=2))

In [ ]:
import matplotlib.pyplot as plt

qualitative_dir = RUN_DIR / 'qualitative_predictions'
qualitative_dir.mkdir(exist_ok=True)
batch = next(iter(test_loader))
with torch.no_grad():
    predictions = best_model(batch['rgb'].to(DEVICE)).cpu()
for index in range(min(8, len(predictions))):
    image = batch['rgb'][index, -1].permute(1, 2, 0).numpy()
    image = np.clip(image * np.asarray(preprocessing.rgb_std) + np.asarray(preprocessing.rgb_mean), 0, 1)
    target, prediction = batch['bc_target'][index], predictions[index]
    figure, axis = plt.subplots(figsize=(7, 4)); axis.imshow(image); axis.axis('off')
    axis.set_title(f'target [steer, throttle] = {target.tolist()}\nprediction = {prediction.tolist()}\nabsolute error = {(prediction-target).abs().tolist()}')
    figure.savefig(qualitative_dir / f'prediction_{index:03d}.png', bbox_inches='tight'); plt.close(figure)
print('saved:', qualitative_dir)

## 8. Condition breakdown and artifact export

Condition metrics should be populated from the held-out metadata and, when available, the Phase 2 scenario-memory artifacts. Empty or very small subsets must be marked non-informative rather than reported as generalization claims.

In [ ]:
def coarse_condition(metadata):
    precipitation = float(metadata.get('weather_precipitation') or 0.0)
    fog = float(metadata.get('weather_fog_density') or 0.0)
    sun = float(metadata.get('weather_sun_altitude_angle') or 0.0)
    speed = float(metadata.get('speed_kmh') or 0.0)
    nearby = int(metadata.get('nearby_vehicles_50m') or 0)
    return {'weather': 'adverse' if precipitation > 10 or fog > 10 else 'clear', 'lighting': 'low_sun' if sun < 10 else 'daylight', 'speed': 'high_speed' if speed >= 40 else 'low_speed', 'traffic': 'dense' if nearby >= 10 else 'light'}

from collections import defaultdict
condition_batches = defaultdict(list)
best_model.eval()
with torch.no_grad():
    for condition_batch in test_loader:
        condition_predictions = best_model(condition_batch['rgb'].to(DEVICE)).cpu()
        for index, metadata in enumerate(condition_batch['metadata']):
            for family, bucket in coarse_condition(metadata).items():
                condition_batches[(family, bucket)].append((condition_predictions[index], condition_batch['bc_target'][index]))
condition_metrics = {'phase2_caveat': 'Occupancy disagreement is a heuristic association, not causal evidence.', 'conditions': {}}
for (family, bucket), pairs in condition_batches.items():
    key = f'{family}:{bucket}'
    condition_metrics['conditions'][key] = {'sample_count': len(pairs), 'informative': len(pairs) >= 20}
    if len(pairs) >= 20:
        condition_metrics['conditions'][key].update(action_metrics(torch.stack([pair[0] for pair in pairs]), torch.stack([pair[1] for pair in pairs])))
(RUN_DIR / 'condition_metrics.json').write_text(json.dumps(condition_metrics, indent=2))
artifact_manifest = {'commit': COMMIT, 'run_dir': str(RUN_DIR), 'drive_run_dir': str(DRIVE_RUN_DIR), 'best_checkpoint': str(DRIVE_RUN_DIR / 'best_checkpoint.pt'), 'test_metrics': str(RUN_DIR / 'test_metrics.json'), 'condition_metrics': str(RUN_DIR / 'condition_metrics.json')}
(RUN_DIR / 'artifact_manifest.json').write_text(json.dumps(artifact_manifest, indent=2))
print(json.dumps(artifact_manifest, indent=2))